# Activité Pratique N°3 — RAG Multi Modal Agentique
## Étape 2 — Vectorisation dans un espace latent commun (embeddings CLIP)

**IA Générative Multi Modale — 5IAD G1, EMSI** · Étudiant : BENHAYOUN SADAFI Mohamed

**Objectif de cette étape :** charger CLIP (ViT-B-32, poids laion2b_s34b_b79k) pour représenter textes et images dans le même espace, avec des vecteurs normalisés.

Document source : *Attention Is All You Need* (`pdfs/attention is all you need.pdf`).

**CLIP** (*Contrastive Language–Image Pre-training*) apprend un **espace latent commun** où une image est proche de sa description.
- `ViT-B-32` : encodeur d'images Vision Transformer, variante Base, patchs de 32 × 32 pixels ;
- `laion2b_s34b_b79k` : entraîné sur ~2 milliards de paires image–texte (LAION), 34 milliards d'exemples vus, lots de 79 000 paires.

> 🔁 **Code refait** — la configuration des chemins, repris de l'étape 1. Chaque notebook est indépendant et recrée ce dont il a besoin.

In [ ]:
import base64
import hashlib
import json
import os
import re
from io import BytesIO
from pathlib import Path

from IPython.display import Markdown, display
from PIL import Image

RACINE = Path.cwd()
PDF_PATH = RACINE / "pdfs" / "attention is all you need.pdf"
OUTPUT_DIR = RACINE / "resultats_pdf"
TEXT_DIR = OUTPUT_DIR / "textes"
IMAGE_DIR = OUTPUT_DIR / "images"
CHROMA_DIR = RACINE / "chroma_db"
COLLECTION_NOM = "pdf_clip_vit_b32"
IMAGE_REQUETE = RACINE / "images_requetes" / "architecture_transformer.png"
print("PDF :", PDF_PATH.name, "| existe :", PDF_PATH.is_file())

In [ ]:
import open_clip
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(DEVICE).eval()
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
print("Appareil :", DEVICE)
print("Prétraitement des images :", preprocess)

### Encoder un texte et une image

Textes → tokenisation → encodage CLIP → **normalisation L2** → listes Python (format attendu par ChromaDB).
Images → redimensionnement, recadrage central, tenseur `[3, 224, 224]` normalisé → encodage CLIP → normalisation L2.

In [ ]:
@torch.inference_mode()
def encoder_textes(textes):
    vecteurs = model.encode_text(tokenizer(textes).to(DEVICE)).float()
    vecteurs = vecteurs / vecteurs.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return vecteurs.cpu().tolist()


@torch.inference_mode()
def encoder_image(image):
    if isinstance(image, (str, Path)):
        with Image.open(image) as img:
            tenseur = preprocess(img.convert("RGB"))
    else:
        tenseur = preprocess(image.convert("RGB"))
    print("  tenseur image :", tuple(tenseur.shape))
    vecteur = model.encode_image(tenseur.unsqueeze(0).to(DEVICE)).float()
    vecteur = vecteur / vecteur.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return vecteur.cpu().tolist()[0]


vecteur_texte = encoder_textes(["the Transformer model architecture with encoder and decoder"])[0]
vecteur_image = encoder_image(IMAGE_DIR / "page_003_image_94.png")
print("Dimension texte :", len(vecteur_texte), "| norme :", round(float(torch.tensor(vecteur_texte).norm()), 4))
print("Dimension image :", len(vecteur_image), "| norme :", round(float(torch.tensor(vecteur_image).norm()), 4))
print("5 premières valeurs (texte) :", [round(v, 4) for v in vecteur_texte[:5]])

### Vérification : textes et images sont comparables

Les vecteurs étant normalisés, la **similarité cosinus** est un simple produit scalaire.
On compare les 3 figures du PDF à plusieurs descriptions : chaque figure doit être plus proche de « sa » description.

In [ ]:
descriptions = [
    "the Transformer model architecture with encoder and decoder stacks",
    "scaled dot-product attention diagram",
    "multi-head attention diagram with several parallel attention layers",
    "a photograph of a cat",
]
figures = sorted(IMAGE_DIR.iterdir())
T = torch.tensor(encoder_textes(descriptions))
I = torch.tensor([encoder_image(f) for f in figures])
similarites = I @ T.T

print(" " * 26 + " | ".join(f"desc {j + 1}" for j in range(len(descriptions))))
for f, ligne in zip(figures, similarites):
    print(f"{f.name:26s}" + " | ".join(f"{v:6.3f}" for v in ligne.tolist()))
for j, d in enumerate(descriptions, 1):
    print(f"desc {j} : {d}")

**Lecture :** les similarités texte–image de CLIP sont faibles en valeur absolue (souvent 0,1 à 0,35), mais c'est leur **classement** qui compte.
La description « chat » doit rester la moins proche des trois figures ; les schémas techniques, très éloignés des photos de LAION, sont plus difficiles à distinguer entre eux.